# Comparación preliminar de estrategias de inicialización para AA

Este notebook compara tres estrategias de inicialización del análisis de arquetipos: uniforme, *Furthest Sum* y PCA con envolvente convexa y *Furthest Sum*.

La comparación corresponde a una prueba preliminar realizada con `interpolacion_50`, normalización min-max, \(K=2\) y semilla 0. Su propósito es analizar si las estrategias alcanzan soluciones comparables y observar su comportamiento de convergencia antes de continuar con la selección secuencial de la normalización, la representación y el número de arquetipos.

El conjunto de test no interviene en esta comparación.

In [1]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_proyecto():
    """Buscar la raíz del proyecto desde el directorio actual."""

    ruta_actual = Path.cwd().resolve()

    for ruta_candidata in (
        ruta_actual,
        *ruta_actual.parents,
    ):
        tiene_src = (
            ruta_candidata
            / "src"
        ).is_dir()

        tiene_notebooks = (
            ruta_candidata
            / "notebooks"
        ).is_dir()

        if tiene_src and tiene_notebooks:
            return ruta_candidata

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


RUTA_PROYECTO = encontrar_raiz_proyecto()

RUTA_REGISTRO = (
    RUTA_PROYECTO
    / "resultados"
    / "registro_experimentos.csv"
)


if not RUTA_REGISTRO.exists():
    raise FileNotFoundError(
        f"No se encontró el registro:\n{RUTA_REGISTRO}"
    )


registro = pd.read_csv(
    RUTA_REGISTRO
)


print(
    "Raíz del proyecto:",
    RUTA_PROYECTO,
)

print(
    "Ejecuciones registradas:",
    len(registro),
)

print(
    "Ejecuciones de AA:",
    (
        registro["modelo"]
        == "archetypal_analysis"
    ).sum(),
)

Raíz del proyecto: C:\Users\adanm\Code\tesis
Ejecuciones registradas: 1094
Ejecuciones de AA: 14


In [2]:
# Columnas relevantes para comparar las inicializaciones
columnas_comparacion = [
    "experiment_id",
    "inicializacion",
    "iteraciones_reales",
    "convergio",
    "loss_final",
    "mse_train",
    "mse_val",
    "rss_val",
    "purity_val",
    "davies_bouldin_val",
    "silhouette_val",
    "ami_val",
    "tiempo_entrenamiento_segundos",
]


# Recuperar las tres ejecuciones de la prueba preliminar
comparacion_inicializaciones = (
    registro.loc[
        (registro["modelo"] == "archetypal_analysis")
        & (registro["dataset"] == "interpolacion_50")
        & (registro["normalizacion"] == "minmax")
        & (registro["K"] == 2)
        & (registro["seed_modelo"] == 0),
        columnas_comparacion,
    ]
    .sort_values("inicializacion")
    .reset_index(drop=True)
)


inicializaciones_esperadas = {
    "uniform",
    "furthest_sum",
    "pca_convex_hull_furthest_sum",
}


# Comprobar que la comparación esté completa
if len(comparacion_inicializaciones) != 3:
    raise ValueError(
        "Se esperaban exactamente tres ejecuciones "
        "para comparar las inicializaciones."
    )


if set(
    comparacion_inicializaciones["inicializacion"]
) != inicializaciones_esperadas:
    raise ValueError(
        "Las inicializaciones encontradas no coinciden "
        "con las tres estrategias esperadas."
    )


if not comparacion_inicializaciones["convergio"].all():
    raise ValueError(
        "Al menos una inicialización no alcanzó convergencia."
    )


display(
    comparacion_inicializaciones[
        [
            "inicializacion",
            "iteraciones_reales",
            "convergio",
            "mse_val",
            "purity_val",
            "davies_bouldin_val",
            "silhouette_val",
            "ami_val",
            "tiempo_entrenamiento_segundos",
        ]
    ]
)

,inicializacion,iteraciones_reales,convergio,mse_val,purity_val,davies_bouldin_val,silhouette_val,ami_val,tiempo_entrenamiento_segundos
0,furthest_sum,20,True,0.014944,0.903480,1.336458,0.344380,0.576365,106.790368
1,pca_convex_hull_furthest_sum,16,True,0.014944,0.903528,1.336447,0.344451,0.576504,90.029007
2,uniform,22,True,0.014944,0.903480,1.336458,0.344380,0.576365,128.283531


## Resultado de la comparación

Las tres estrategias de inicialización alcanzaron la convergencia y produjeron resultados prácticamente equivalentes en validación. El MSE fue el mismo con la precisión mostrada, mientras que las diferencias en pureza, Davies–Bouldin, silhouette y AMI fueron inferiores a \(0.0002\).

La inicialización `pca_convex_hull_furthest_sum` necesitó 16 iteraciones, frente a 20 de *Furthest Sum* y 22 de la inicialización uniforme. En esta ejecución también presentó el menor tiempo de entrenamiento, con 90.03 segundos, en comparación con 106.79 y 128.28 segundos, respectivamente.

Dado que las tres estrategias alcanzaron una solución de calidad equivalente, se seleccionó `pca_convex_hull_furthest_sum` como decisión operativa por su menor número de iteraciones en esta prueba y por su criterio geométrico de selección de candidatos extremos. Esta decisión no implica que la estrategia sea universalmente superior: la comparación corresponde a una prueba preliminar realizada con `interpolacion_50`, normalización min-max, \(K=2\) y una sola semilla.

La estabilidad de la configuración final se evalúa posteriormente mediante ejecuciones con distintas semillas.